# Google Stock Price Prediction using Simple RNN

This notebook presents the cleaned, portfolio-ready workflow for predicting the **next trading session's Google closing price**.

> **Financial disclaimer:** Educational and portfolio demonstration only. This is not financial advice and must not be used for investment decisions.

The improved model predicts the next-day **log return** from the previous 10 daily log returns and converts that estimate back into a closing price. This is more stable than asking a tanh Simple RNN to extrapolate raw price levels far beyond the training range.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import load_and_standardize
from src.feature_engineering import add_diagnostic_features, build_return_forecast_frame
from src.sequence_generation import create_return_sequences, chronological_split_and_scale
from src.model_training import build_simple_rnn_model
from src.forecasting_pipeline import train_evaluate_pipeline

DATA_PATH = PROJECT_ROOT / 'data' / 'sample_google_stock_data.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
MODEL_DIR = PROJECT_ROOT / 'models'
WINDOW_SIZE = 10

## 1. Load and validate the sample

In [ ]:
stock_df, target_column, quality_report = load_and_standardize(DATA_PATH, 'Close')
print(json.dumps(quality_report, indent=2))
stock_df.head()

## 2. Explore the time series

In [ ]:
diagnostic_df = add_diagnostic_features(stock_df, target_column)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(diagnostic_df['Date'], diagnostic_df[target_column], label='Close')
ax.plot(diagnostic_df['Date'], diagnostic_df['Moving_Average_20'], label='20-day moving average')
ax.set_title('Google closing-price trend')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.show()

## 3. Create return sequences and chronological splits

In [ ]:
forecast_frame = build_return_forecast_frame(stock_df, target_column)
sequences = create_return_sequences(forecast_frame, window_size=WINDOW_SIZE)
split = chronological_split_and_scale(sequences)

print('Sequence shape:', sequences.X.shape)
print('Split sizes:', split.split_indices)
print('Input meaning: previous 10 daily log returns')
print('Target meaning: next-day log return')

## 4. Inspect the Simple RNN architecture

In [ ]:
model = build_simple_rnn_model(window_size=WINDOW_SIZE, n_features=1)
model.summary()

## 5. Optional retraining

The repository already contains a trained model and curated outputs. Set `RUN_TRAINING=True` to regenerate them.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    results = train_evaluate_pipeline(
        data_path=DATA_PATH,
        project_root=PROJECT_ROOT,
        target_column='Close',
        window_size=WINDOW_SIZE,
    )
    print(results['metrics'])
else:
    print('Using the saved portfolio artifacts. Run train_model.py to retrain from the command line.')

## 6. Review saved test results and baselines

In [ ]:
metrics = json.loads((OUTPUT_DIR / 'model_metrics.json').read_text())
comparison = pd.read_csv(OUTPUT_DIR / 'model_comparison.csv')
predictions = pd.read_csv(OUTPUT_DIR / 'test_predictions.csv', parse_dates=['Date'])

print(json.dumps(metrics, indent=2))
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(predictions['Date'], predictions['Actual_Close'], label='Actual')
ax.plot(predictions['Date'], predictions['Predicted_Close'], label='Simple RNN')
ax.set_title('Untouched test period: actual vs predicted close')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.show()

## 7. Next-session demonstration forecast

In [ ]:
next_forecast = json.loads((OUTPUT_DIR / 'next_day_forecast.json').read_text())
next_forecast

## Interpretation

- The test metrics evaluate the final 15% of observations after a 70% train and 15% validation split.
- Scalers are fit only on training observations.
- Training uses `shuffle=False` to preserve temporal order.
- The naive previous-close baseline is intentionally difficult to beat for daily stock prices.
- The model is a demonstration of sequential modeling, not a trading system.

**Financial disclaimer:** Stock prices depend on news, macroeconomic conditions, market microstructure, investor behavior, and other information that historical prices alone do not capture.